In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

ModuleNotFoundError: No module named 'pandas'

In [3]:
base_model = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B" 

"""
alternative fine tuned version:
model_id = "abhi9ab/DeepSeek-R1-Distill-Llama-8B-finance-v1"
"""

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Make my own fine-tuned Lora-Adapter
# model = PeftModel.from_pretrained(model, "./your-trained-lora-adapter")

model = model.eval()
print("model loaded!")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 77.91it/s]


model loaded!


In [11]:
for headline in test_headlines:
    prompt = f"<｜begin of sentence｜><｜User｜>Analyze the market sentiment of this news: {headline}<｜Assistant｜><｜thought｜>"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024,
            temperature=0.1,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    clean_output = full_output.replace('Ġ', ' ').replace('Ċ', '\n').replace('pre', '')
    
    print(f"\nFINANCIAL ANALYSIS FOR: {headline}")
    print("-" * 50)
    
    if "Result" in clean_output:
        parts = clean_output.split("Result")
        print(f"REASONING: {parts[0].strip()}")
        print(f"FINAL SENTIMENT: Result {parts[1].strip()}")
    else:
        print(clean_output)
    
    print("="*50)


FINANCIAL ANALYSIS FOR: NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026.
--------------------------------------------------
<beginofsentence><｜User｜>Analyzethemarketsentimentofthisnews:NVIDIAbeatsQ4earningsexpectationsbutwarnsofsupplychainconstraintsin2026.<｜Assistant｜><thought>Alright, so I need to analyze the market sentiment of this news: NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026. Hmm, okay, let's break this down step by step. First, I should understand what the news is saying. NVIDIA is a big company in the tech sector, especially in graphics cards and GPUs. They reported their Q4 earnings, which were better than what the market was expecting. That's a positive sign. But then they also warned about supply chain constraints in 2026. That's a bit concerning because supply chain issues can affect their ability to produce products and meet demand.

So, how does this affect the market sentiment? Well, when a 

In [12]:
for headline in test_headlines:
    prompt = f"""
        You are a financial sentiment classifier.
        
        Analyze the headline and respond in EXACTLY this format:
        
        Sentiment: [Bullish, Bearish, or Neutral]
        Confidence: [number between 0 and 1]
        
        Rules:
        - Do NOT explain your reasoning
        - Output ONLY the two lines above
        - Be decisive
        
        Headline: {headline}
        """
            
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024,
            temperature=0.1,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    clean_output = full_output.replace('Ġ', ' ').replace('Ċ', '\n').replace('pre', '')
    
    print(f"\nFINANCIAL ANALYSIS FOR: {headline}")
    print("-" * 50)
    
    if "Result" in clean_output:
        parts = clean_output.split("Result")
        print(f"REASONING: {parts[0].strip()}")
        print(f"FINAL SENTIMENT: Result {parts[1].strip()}")
    else:
        print(clean_output)
    
    print("="*50)


FINANCIAL ANALYSIS FOR: NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026.
--------------------------------------------------
Youareafinancialsentimentclassifier.AnalyzetheheadlineandrespondinEXACTLYthisformat:Sentiment:[Bullish,Bearish,orNeutral]Confidence:[numberbetween0and1]Rules:-DoNOTexplainyourreasoning-OutputONLYthetwolinesabove-BedecisiveHeadline:NVIDIAbeatsQ4earningsexpectationsbutwarnsofsupplychainconstraintsin2026.BedecisiveHeadline:NVIDIAbeatsQ4earningsexpectationsbutwarnsofsupplychainconstraintsin20 26.

Okay, so I need to analyze the headline "NVIDIA beats Q4 earnings expectations but warns of supply chain constraints in 2026." I'm supposed to determine the sentiment as either Bullish, Bearish, or Neutral, and assign a confidence score between 0 and 1. 

First, let's break down the headline. NVIDIA is a major player in the tech industry, especially in GPUs and AI. Beating earnings expectations is generally a positive sign because it show